# LangChain Agent Benchmark 01: Models, Prompts, Context, and Outputs

This notebook benchmarks model and prompt variables while keeping the task constant. It uses OpenRouter chat completions with free model variants.

How to use it: for every model's response, follow the reflections points to give a 1-5 rating for each model.
At the end of the notebook, sum your answers to see how the models fare on these biological tasks.

In [ ]:

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "openai",
    "matplotlib",
    "pexpect",
])

# restart kernel after running

0

In [ ]:
# Verify correct installation
import openai

print(openai.__version__)
print(openai.__file__)

## OpenRouter free-model limits

Rate limits govern how many requests you can make. There are a few rate limits that apply to certain types of requests, regardless of account status:

**Free usage limits**: When using a free model variant (with an ID ending in `:free`),as we are in this tutorial, the following limits apply:

| Credits purchased (all time) | Requests per minute | Requests per day |
|---|---:|---:|
| Less than 10 | 20 | 50 |
| At least 10 | 20 | 1000 |

Keep an eye on your usage here: https://openrouter.ai/activity/

**DDoS protection**: Cloudflare's DDoS protection will block requests that dramatically exceed reasonable usage.

See OpenRouter's limits documentation: https://openrouter.ai/docs/api_reference/limits#handling-429-errors


## 0. Library import and global variable definition

Before running this code cell, make sure you have set up your `.env` file with `OPENROUTER_API_KEY` and `GOOGLE_API_KEY`. This contains your personal credentials and is read with `os.getenv()`.

In [29]:
from dotenv import load_dotenv
import os
import time
from typing import Literal

import pandas as pd
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv("../.env", override=True)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
# OPENROUTER_SITE_URL = os.getenv("OPENROUTER_SITE_URL")
# OPENROUTER_APP_NAME = os.getenv("OPENROUTER_APP_NAME", "LangChain Agent Benchmark")

# Free OpenRouter models allow 20 requests/minute. A small pause keeps full-notebook runs below that limit.
REQUEST_PAUSE_SECONDS = 3

MODEL_SPECS = [
    {
        "model_type": "OpenAI GPT-OSS 20B free model",
        "model": "openai/gpt-oss-20b:free",
    },
    {
        "model_type": "NVIDIA Nemotron Nano Omni free reasoning model",
        "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    },
    {
        "model_type": "Google Gemma 4 31B: instruction tuned, dense",
        "model": "google/gemma-4-31b-it:free",
    },
    {
        "model_type": "Google Gemma 4 26B A4B, instruction model, MoE",
        "model": "google/gemma-4-26b-a4b-it:free",
    },
]
DEFAULT_MODEL = MODEL_SPECS[0]
DEFAULT_MODEL_ID = DEFAULT_MODEL["model"]

if OPENROUTER_API_KEY:
    print("OpenRouter API key loaded.")
else:
    print("Set OPENROUTER_API_KEY before running the examples.")


TimeoutError: [Errno 60] Operation timed out

### Create the OpenRouter client object
The following code creates the client object that will send requests to OpenRouter, it takes these parameters:
- The api_key authenticates our requests,
- base_url redirects the client from OpenAI’s default endpoint to OpenRouter,
- default_headers adds optional app/site metadata for OpenRouter tracking.

This does not call a model yet. It only configures the connection.

In [27]:
# default_headers = {"X-Title": OPENROUTER_APP_NAME}
# if OPENROUTER_SITE_URL:
#     default_headers["HTTP-Referer"] = OPENROUTER_SITE_URL

OPENROUTER_CLIENT = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    # default_headers=default_headers,
)


## 1. LLM choice

LLM choice affects factual accuracy, biological knowledge, reasoning quality, hallucination behavior, speed, verbosity, and code quality. This section isolates several model-level variables one at a time where the available free OpenRouter models allow it.


### Different vendors

Different labs make different modeling choices: training data, instruction tuning, safety behavior, output style, and evaluation targets. This block keeps the task constant and compares one free model from OpenAI, NVIDIA, and Google.

**Reflection Prompts**
- Compare which answers seem most reliable and identify which biological details make you think so.
- Separate content quality from style: is a more fluent answer also more correct?
- Note whether one vendor consistently gives more caveats, stronger claims, or clearer uncertainty.


In [28]:
question = 'Explain why batch correction matters in single-cell RNA-seq analysis. Keep the answer under 120 words.'
vendor_models = [
    {"vendor": "OpenAI", "model": "openai/gpt-oss-20b:free"},
    {"vendor": "NVIDIA", "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"},
    {"vendor": "Google", "model": "google/gemma-4-31b-it:free"},
]

for spec in vendor_models:
    heading = spec['vendor']
    chunks = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=[{"role": "user", "content": question}],
        temperature=0,
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices and chunk.choices[0].delta.content:
            chunks.append(chunk.choices[0].delta.content)
            markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
            answer_display.update(Markdown(markdown_text))
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### OpenAI
`openai/gpt-oss-20b:free`

Batch correction is essential in single‑cell RNA‑seq because technical factors (library prep, sequencing run, reagent lots) introduce systematic differences that can overwhelm true biology. Without correction, cells from different batches cluster together, masking real cell types or states, inflating false positives in differential expression, and biasing trajectory or network inference. Proper batch integration removes these artifacts, aligns shared cell populations across experiments, and preserves

KeyboardInterrupt: 

### Number of parameters

This block keeps the comparison inside the Nemotron family so the parameter-size discussion is less confounded by vendor. It compares a small Nemotron model, a Nano Omni model, and a larger Super model. The models still differ in architecture and modality support, so treat this as a practical size comparison rather than a perfectly controlled ablation.

**Reflection Prompts**
- Does the larger model give a more specific or more cautious biological answer?
- Compare latency and token use against answer quality: is the larger model worth it here?
- Look for whether extra scale improves scientific precision or mainly changes style.


In [19]:
question = 'Interpret high FKBP5 and CRISPLD2 expression after dexamethasone treatment. Keep the answer under 120 words.'
parameter_models = [
    {"parameter_label": "9B", "model": "nvidia/nemotron-nano-9b-v2:free"},
    # {"parameter_label": "30B stored, ~3B active", "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"},
    {"parameter_label": "120B stored, ~12B active", "model": "nvidia/nemotron-3-super-120b-a12b:free"},
]

for spec in parameter_models:
    heading = spec['parameter_label']
    chunks = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=[{"role": "user", "content": question}],
        temperature=0,
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices and chunk.choices[0].delta.content:
            chunks.append(chunk.choices[0].delta.content)
            markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
            answer_display.update(Markdown(markdown_text))
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### 9B
`nvidia/nemotron-nano-9b-v2:free`



High FKBP5 and CRISPLD2 expression post-dexamethasone may indicate altered glucocorticoid signaling or stress responses. FKBP5 modulates GR function, suggesting potential glucocorticoid resistance. CRISPLD2, linked to immune regulation, might reflect immune activation or cellular stress. Together, they could signal a complex adaptive response to dexamethasone, possibly involving inflammation or metabolic changes. Further context is needed for precise interpretation.


seconds: 6.189
prompt_tokens: 38
completion_tokens: 295
total_tokens: 333
reasoning_tokens: 273
cost: 0


### 120B stored, ~12B active
`nvidia/nemotron-3-super-120b-a12b:free`

Dexamethasone activates the glucocorticoid receptor (GR), which directly induces transcription of FKBP5, a co‑chaperone that modulates GR sensitivity and provides a negative‑feedback loop to limit glucocorticoid signaling. Elevated FKBP5 therefore reflects successful GR activation and a cellular attempt to attenuate further steroid signaling. CRISPLD2 is also a glucocorticoid‑responsive secreted protein implicated in extracellular‑matrix remodeling and anti‑inflammatory pathways; its up‑regulation suggests dexamethasone‑driven modulation of tissue‑repair and inflammation‑resolving processes. Together, high FKBP5 and CRISPLD2 expression indicate that dexamethasone has effectively engaged GR‑mediated transcriptional programs that promote feedback inhibition and tissue‑protective, anti‑inflammatory responses.

seconds: 7.585
prompt_tokens: 42
completion_tokens: 282
total_tokens: 324
reasoning_tokens: 171
cost: 0


### Different LLM architecture

A dense model uses all of its stored parameters for every token, making behavior more direct but often more compute-heavy. A Mixture of Experts (MoE) model routes each token through a subset of parameters (called "experts"), which can reduce compute and latency but may change reproducibility and style. This block compares regular Gemma 4 against its MoE counterpart.

**Reflection Prompts**
- Does dense versus MoE change answer quality, or mostly latency and token behavior?
- Look for instability or oddly different framing between architectures.
- Decide which architecture you would choose for a repeated scientific-note workflow.


In [21]:
question = 'A T-cell cluster has high interferon-stimulated genes after stimulation. Explain the biological interpretation and two checks. Keep the answer under 120 words.'
# architecture_models = [
#     {"architecture": "dense", "model": "google/gemma-4-31b-it:free"},
#     {"architecture": "MoE", "model": "google/gemma-4-26b-a4b-it:free"},
# ]
architecture_models = [
    {"architecture": "dense", "model": "nvidia/nemotron-nano-12b-v2-vl:free"},
    {"architecture": "MoE", "model": "nvidia/nemotron-3-nano-30b-a3b:free"},
]

for spec in architecture_models:
    heading = spec['architecture']
    chunks = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=[{"role": "user", "content": question}],
        temperature=0,
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices and chunk.choices[0].delta.content:
            chunks.append(chunk.choices[0].delta.content)
            markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
            answer_display.update(Markdown(markdown_text))
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### dense
`nvidia/nemotron-nano-12b-v2-vl:free`

High interferon-stimulated genes (ISGs) in T-cells post-stimulation indicate robust activation of the interferon signaling pathway, typically triggered by viral infections, immune activation, or inflammatory stimuli. This suggests the T-cells are engaged in antiviral defense or adaptive immune responses.  

**Checks:**  
1. **Gene-specific validation:** Confirm ISG expression (e.g., *MX1*, *ISG15*) via qPCR or RNA-seq to rule out off-target effects.  
2. **Functional correlation:** Assess IFN-γ production or STAT1 phosphorylation to validate upstream signaling activity.  

This combination ensures observed ISG upregulation reflects genuine interferon-driven activation.


seconds: 8.93
prompt_tokens: 44
completion_tokens: 136
total_tokens: 180
reasoning_tokens: 0
cost: 0


### MoE
`nvidia/nemotron-3-nano-30b-a3b:free`

**Biological interpretation**  
A T‑cell cluster showing strong interferon‑stimulated gene (ISG) expression after stimulation indicates that the cells have been driven into an “interferon‑responsive” state. This typically reflects activation of the JAK‑STAT pathway downstream of type I or type II interferon receptors, leading to transcription of antiviral, immunomodulatory, and maturation genes (e.g., MX1, ISG15, STAT1). In T cells, such a signature often marks heightened readiness to respond to viral or inflammatory cues, possible differentiation toward a more effector or exhausted phenotype, and may be linked to chronic stimulation or a tumor‑microenvironment context.

**Two checks**  
1. **Stimulus specificity** – Verify that the stimulus (e.g., IFN‑α/β, viral mimic, cytokine cocktail) truly engages interferon receptors and that the observed ISG up‑regulation is not a downstream artifact of general activation (e.g., NF‑κB). Compare with unstimulated controls and, if possible, with a non‑interferon stimulus.  
2. **Cell‑type purity & state** – Confirm that the high‑ISG signal originates from genuine T cells and not contaminating myeloid or NK cells, and assess whether the cells are naïve, memory, or differentiated (e.g., via surface markers CD3/CD4/CD8, activation markers CD69, CD25). Single‑cell clustering or flow cytometry can resolve this.

seconds: 3.709
prompt_tokens: 46
completion_tokens: 345
total_tokens: 391
reasoning_tokens: 42
cost: 0


### Reasoning vs non-reasoning

Reasoning models spend extra internal tokens before producing the final answer. This is one of the biggest conceptual gaps in current LLM use because it can change latency, token use, and accuracy.

Use OpenRouter's `reasoning` request parameter. With the OpenAI Python SDK, safest is to pass it through `extra_body`.

- `reasoning.enabled: True` enables reasoning with default settings.
- `reasoning.effort: "minimal" | "low" | "medium" | "high" | "xhigh" | "max"` controls reasoning budget where supported.
- `reasoning.effort: "none"` disables reasoning where supported.
- `reasoning.exclude: True` lets the model reason internally but hides reasoning tokens from the response.
- Returned reasoning appears in `choices[].message.reasoning` or `choices[].message.reasoning_details`.

OpenRouter docs: [reasoning tokens](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens), [API parameters](https://www.openrouter.ai/docs/api/reference/parameters).

**Reflection Prompts**
- Does explicit reasoning improve the scientific caution or just make the response longer/slower?
- Compare `reasoning_tokens`, latency, and final-answer quality.
- Decide when reasoning is worth the extra cost for biological interpretation tasks.


In [22]:
question = "A sample has high mitochondrial RNA, low detected genes, and elevated IFIT1. Is this a dying-cell population or interferon response? Give a cautious interpretation."
reasoning_runs = [
    {
        "mode": "reasoning on (high effort)",
        "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
        "messages": [{"role": "user", "content": question}],
        "extra_body": {
            "reasoning": {
                "enabled": True,
                "effort": "xhigh",   # minimal, low, medium, high, xhigh, max
                "exclude": False,     # return reasoning if the model/provider exposes it
            }
        },
    },
    {
        "mode": "reasoning off / final answer only",
        "model": "nvidia/nemotron-nano-9b-v2:free",
        "messages": [
            {"role": "system", "content": "Answer directly. Do not include intermediate reasoning."},
            {"role": "user", "content": question},
        ],
        "extra_body": {
            "reasoning": {
                "effort": "none",
                "exclude": True,
            }
        },
    },
]

for spec in reasoning_runs:
    heading = spec["mode"]
    chunks = []
    reasoning_chunks = []
    reasoning_details = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=spec["messages"],
        temperature=0,
        extra_body=spec["extra_body"],
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices:
            delta = chunk.choices[0].delta
            delta_dict = delta.model_dump()
            if delta.content:
                chunks.append(delta.content)
                markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
                answer_display.update(Markdown(markdown_text))
            if delta_dict.get("reasoning"):
                reasoning_chunks.append(delta_dict["reasoning"])
            if delta_dict.get("reasoning_details"):
                reasoning_details.append(delta_dict["reasoning_details"])
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    reasoning = "".join(reasoning_chunks) or None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("reasoning:", reasoning)
    print("reasoning_details:", reasoning_details or None)
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### reasoning on (high effort)
`nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free`

**Cautious interpretation**

- **High mitochondrial RNA** – Mitochondrial transcripts tend to accumulate when cells lose membrane integrity (e.g., necrosis, late‑apoptosis) or when RNA from damaged organelles “leaks” into the cytoplasmic pool. In a pure dying‑cell population you would expect a pronounced mtRNA signal together with a general collapse of nuclear gene expression.

- **Low detected genes** – Fewer genes being captured can arise from several sources: (1) genuine shutdown of transcription (as in terminally differentiated or dying cells), (2) poor RNA quality (RNA degradation, fragmentation) that preferentially leaves abundant, stable mtRNAs while removing many nuclear transcripts, or (3) technical bias (e.g., low input material, library‑prep issues). Thus low gene counts alone do not prove cell death; they simply indicate reduced overall transcriptional output or compromised RNA integrity.

- **Elevated IFIT1** – IFIT1 is a classic interferon‑stimulated gene (ISG). Its up‑regulation points to an active interferon‑signaling cascade, which can be triggered by viral infection, cytokine exposure, or other stress signals. Importantly, interferon responses can occur in both viable and dying cells; dying cells can release mitochondrial RNA that fuels further immune activation.

**Putting it together**

The pattern—*high mtRNA + low nuclear gene detection* **plus** *high IFIT1*—suggests a sample that is **not purely a dying‑cell population** nor a clean interferon‑response signature. The most parsimonious view is that the sample contains a **subpopulation of cells undergoing stress or death** (hence the mtRNA surge and reduced nuclear transcription) **while simultaneously mounting an interferon‑driven response** (IFIT1 up‑regulation).  

Alternatively, the high mtRNA could be a technical artifact (e.g., RNA degradation releasing mitochondrial transcripts) that coincidentally coincides with an interferon response in the remaining viable cells.

**Bottom line:** The data are compatible with a mixed or transitional state—partly dying cells releasing mitochondrial RNA, and partly cells actively responding to interferon. Additional controls (e.g., cell‑viability dyes, spike‑in RNAs, mitochondrial DNA content, or protein‑level markers) would be needed to disentangle these possibilities.

reasoning: We need to answer: "A sample has high mitochondrial RNA, low detected genes, and elevated IFIT1. Is this a dying-cell population or interferon response? Give a cautious interpretation."

We need to interpret based on typical molecular signatures. High mitochondrial RNA (mtRNA) often indicates cell death or mitochondrial leakage, as mitochondria release RNA when cells die. Low detected genes suggests low overall transcriptional activity, maybe due to cell death or low quality. Elevated IFIT1 is a marker of interferon response, indicating antiviral or inflammatory signaling.

Thus the sample could be a mix: dying cells releasing mtRNA, but also an interferon response. Cautious interpretation: it's likely a mixture; high mtRNA suggests dying cells, low gene detection suggests poor RNA quality or cell loss, while IFIT1 elevation points to interferon activation. So not purely dying cells, nor purely interferon response; could be dying cells with ongoing interferon signaling, or s

### reasoning off / final answer only
`nvidia/nemotron-nano-9b-v2:free`



The combination of high mitochondrial RNA (suggesting cell stress or death), low detected genes (indicating reduced transcriptional activity), and elevated IFIT1 (an interferon-stimulated gene) could reflect either a dying-cell population or an interferon response. However, elevated IFIT1 specifically points toward an interferon response, which may be triggered by cellular stress or damage. The other markers could align with dying cells, but the interferon signature is a key indicator. A cautious interpretation would note that both processes might coexist, but the interferon response is more directly supported by IFIT1 elevation.


reasoning: None
reasoning_details: None
seconds: 29.319
prompt_tokens: 53
completion_tokens: 755
total_tokens: 808
reasoning_tokens: 853
cost: 0


## 2. Temperature

Temperature controls sampling randomness: lower values produce more deterministic answers while higher values increase creativity (can also increase hallucination risk). This works by flattening the output probabilities of the model. 

We show stochasticity by repeating generating the answers multiple times. The key comparison is not only temperature 0 versus 1, but also variation across repeated runs at the same temperature. At temperature 0, repeated answers should be relatively stable; at temperature 1, the selected experiments and framing should vary more.

**Reflection Prompts**
- Compare replicates at the same temperature: which elements remain stable, and which ones change?
- Evaluate whether greater variety produces genuinely more useful ideas or just different wording.
- Decide which temperature you would use for a scientific task and justify the trade-off between creativity and control.


In [ ]:
question = """
The airway bulk RNA-seq experiment identified genes that change expression
after dexamethasone treatment.

Suggest three substantially different follow-up experiments.
Answer in concise bullet points, max 120 words.
"""
rows = []

for temperature in [0, 1.0]:
    for replicate in range(2):
        start = time.perf_counter()

        response = OPENROUTER_CLIENT.chat.completions.create(
            model=DEFAULT_MODEL_ID,
            messages=[{"role": "user", "content": question}],
            temperature=temperature,
        )
        usage = response.usage
        completion_tokens_details = usage.completion_tokens_details
        answer = response.choices[0].message.content
        display(Markdown(f"### temperature={temperature}, replicate={replicate + 1}\n\n{answer}"))

        rows.append({
            "temperature": temperature,
            "replicate": replicate + 1,
            "seconds": round(time.perf_counter() - start, 3),
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "reasoning_tokens": completion_tokens_details.reasoning_tokens,
            "cost": usage.cost,
            "usage": usage,
        })
        time.sleep(REQUEST_PAUSE_SECONDS)
        
df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### temperature=0, replicate=1

- **Single‑cell RNA‑seq** of airway epithelial and immune cells after dexamethasone to resolve cell‑type specific transcriptional shifts and identify rare responsive populations.  
- **CRISPR‑Cas9 loss‑/gain‑of‑function screens** targeting the top up‑ and down‑regulated genes in primary airway epithelial cultures to test their causal role in steroid sensitivity, cytokine release, and barrier integrity.  
- **ATAC‑seq or GR ChIP‑seq** to map chromatin accessibility and glucocorticoid‑receptor binding changes, revealing enhancer dynamics and transcriptional regulatory networks driving the observed bulk RNA changes.

### temperature=0, replicate=2

- **Single‑cell RNA‑seq** of airway epithelium after dexamethasone to resolve cell‑type‑specific transcriptional shifts and identify subpopulations that mediate the response.  
- **CRISPR‑Cas9 loss‑of‑function or overexpression** of the most differentially expressed genes in primary airway cultures, followed by barrier‑function, cytokine‑release, and steroid‑resistance assays to test causality.  
- **ATAC‑seq or ChIP‑seq for the glucocorticoid receptor** to map chromatin accessibility and direct binding sites, linking the observed gene‑expression changes to regulatory elements and co‑factor interactions.

### temperature=1.0, replicate=1

- **Single‑cell RNA‑seq / CITE‑seq** of airway epithelium and infiltrating immune cells before and after dexamethasone to pinpoint which indexes of the bulk signal arise from specific cell subsets and to associate transcriptional changes with surface markers.

- **Omni‑ATAC‑seq or GR ChIP‑seq** of sorted airway cells to map shifts in chromatin accessibility or glucocorticoid‑receptor binding at differentially expressed loci, separating direct steroid targets from secondary downstream genes.

- **CRISPR‑Cas9 loss‑/gain‑of‑function screen** on top candidate genes (e.g., cytokine regulators) followed by assays for epithelial barrier integrity or cytokine release, validating causal roles in corticosteroid responsiveness.

### temperature=1.0, replicate=2

- **Functional validation** – Use CRISPR‑Cas9 or siRNA to knock out the top ∼ 10 dex‑induced genes in primary airway epithelial cells; then assess changes in steroid‑induced suppression of pro‑inflammatory cytokines, barrier músicos, or GR-dependent transcriptional reporters.

- **Single‑cell resolution** – Perform scRNA‑seq (or 10× Chromium) on airway samples treated ± dexamethasone; cluster by cell type, quantify cell‑type‑specific DE, and reconstruct differentiation trajectories to identify heterogeneous glucocorticoid responses.

- **In vivo chromatin‑omics** – In a mouse model of airway inflammation, treat with dexamethasone, then harvest lung tissue for ATAC‑seq or ChIP‑seq (GR, H3K27ac). Integrate with bulk transcript data to map enhancer activation and genome‑wide regulatory rewiring underlying the transcriptional changes.

,temperature,replicate,seconds,prompt_tokens,completion_tokens,total_tokens,reasoning_tokens,cost,usage,answer
0,0.000000,1,23.713000,108,977,1085,834,0,"CompletionUsage(completion_tokens=977, prompt_tokens=108, total_tokens=1085, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=834, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=0, is_byok=False, cost_details={'upstream_inference_cost': 6.9956e-05, 'upstream_inference_prompt_cost': 1.566e-06, 'upstream_inference_completions_cost': 6.839e-05})","- **Single‑cell RNA‑seq** of airway epithelial and immune cells after dexamethasone to resolve cell‑type specific transcriptional shifts and identify rare responsive populations. - **CRISPR‑Cas9 loss‑/gain‑of‑function screens** targeting the top up‑ and down‑regulated genes in primary airway epithelial cultures to test their causal role in steroid sensitivity, cytokine release, and barrier integrity. - **ATAC‑seq or GR ChIP‑seq** to map chromatin accessibility and glucocorticoid‑receptor binding changes, revealing enhancer dynamics and transcriptional regulatory networks driving the observed bulk RNA changes."
1,0.000000,2,19.214000,108,823,931,671,0,"CompletionUsage(completion_tokens=823, prompt_tokens=108, total_tokens=931, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=671, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=0, is_byok=False, cost_details={'upstream_inference_cost': 5.9176e-05, 'upstream_inference_prompt_cost': 1.566e-06, 'upstream_inference_completions_cost': 5.761e-05})","- **Single‑cell RNA‑seq** of airway epithelium after dexamethasone to resolve cell‑type‑specific transcriptional shifts and identify subpopulations that mediate the response. - **CRISPR‑Cas9 loss‑of‑function or overexpression** of the most differentially expressed genes in primary airway cultures, followed by barrier‑function, cytokine‑release, and steroid‑resistance assays to test causality. - **ATAC‑seq or ChIP‑seq for the glucocorticoid receptor** to map chromatin accessibility and direct binding sites, linking the observed gene‑expression changes to regulatory elements and co‑factor interactions."
2,1.000000,1,15.606000,108,635,743,467,0,"CompletionUsage(completion_tokens=635, prompt_tokens=108, total_tokens=743, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=467, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=0, is_byok=False, cost_details={'upstream_inference_cost': 4.6016e-05, 'upstream_inference_prompt_cost': 1.566e-06, 'upstream_inference_completions_cost': 4.445e-05})","- **Single‑cell RNA‑seq / CITE‑seq** of airway epithelium and infiltrating immune cells before and after dexamethasone to pinpoint which indexes of the bulk signal arise from specific cell subsets and to associate transcriptional changes with surface markers. - **Omni‑ATAC‑seq or GR ChIP‑seq** of sorted airway cells to map shifts in chromatin accessibility or glucocorticoid‑receptor binding at differentially expressed loci, separating direct steroid targets from secondary downstream genes. - **CRISPR‑Cas9 loss‑/gain‑of‑function screen** on top candidate genes (e.g., cytokine regulators) followed by assays for epithelial barrier integrity or cytokine release, validating causal roles in corticosteroid responsiveness."
3,1.000000,2,17.986000,108,611,719,405,0,"CompletionUsage(completion_tokens=611, prompt_tokens=108, total_tokens=719, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, au

## 3. System prompt

The system prompt sets high-level behavior such as persona, assumption, criteria for tool usage, and any instruction that should always be followed. This is injected at the first turn of the conversation and is kept in memory across all turns. Implemented via `SystemMessage` inside a `ChatPromptTemplate`. 

**Reflection Prompts**
- Observe how the tone changes when the model receives a role or more specific instructions.
- Identify whether the system prompt improves accuracy as well, or mainly changes the form of the answer.
- Ask which instructions make the answer easier to evaluate from a scientific perspective.


In [23]:
question = "A sample has high mitochondrial RNA and low detected genes. Is it a dying-cell population?"
rows = []

# System prompt: minimal
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question},
]
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=messages,
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"system_prompt": "minimal", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# System prompt: biologist persona
messages = [
    {"role": "system", "content": "You are a molecular biologist who explains concepts clearly to computational biology students."},
    {"role": "user", "content": question},
]
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=messages,
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"system_prompt": "biologist persona", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# System prompt: scientific with strict instructions
messages = [
    {"role": "system", "content": "You are a strict scientific assistant. Separate evidence from speculation, state uncertainty, and avoid unsupported claims."},
    {"role": "user", "content": question},
]
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=messages,
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"system_prompt": "strict scientific", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


KeyboardInterrupt: 

## 4. Prompt template

This is a scaffold that controls how the task is framed and allows the injection of dynamic data like user inputs or variables at runtime. Implemented via `PromptTemplate` or `ChatPromptTemplate`. 

**Reflection Prompts**
- Compare how strongly the prompt structure guides the structure of the final answer.
- Notice whether the few-shot examples help the model imitate the format or also reason better.
- Identify which template makes it easiest to correct or compare answers across groups.


In [24]:
observation = "a T-cell cluster has high interferon-stimulated genes after stimulation"
rows = []

# Free form prompt:  a plain instruction with no examples or required output structure
prompt = f"Explain whether {observation} is biologically meaningful."
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"template": "free form", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# Few-shot prompt: includes an example input-output pair to guide the answer style
prompt = (
    "Example:\n"
    "Observation: high MALAT1 in low-quality nuclei.\n"
    "Answer: This may reflect nuclear RNA content or technical quality; validate with QC metrics and markers.\n\n"
    f"Observation: {observation}\n"
    "Answer:"
)
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"template": "few shot", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# Structured prompt: asks the model to organize its answer into named sections
prompt = (
    f"Observation: {observation}\n"
    "Return sections: Interpretation | Alternative explanations | Checks | Confidence."
)
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"template": "structured", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 5. Context length

LLM answers depend on what is present in the prompt context window. Models with large context windows receive more background information, which can improve grounding but can also introduce distractions, truncation, and irrelevant details. Context is an intrinsic characteristic of an LLM, and should be taken into account when choosing an LLM for your specific purposes. 


**Reflection Prompts**
- Evaluate which information is preserved when the context is short and which only appears with more context.
- Look for signs of distraction: does more context always make the answer better?
- Discuss what minimum context would be sufficient to answer the question responsibly.


In [25]:
base_context = "Protocol note: Samples were PBMCs stimulated with IFN-beta for 6 hours. Mitochondrial reads above 20% were filtered."
small_context = base_context
large_context = "\n".join([base_context] + [
    "Marker note: IFIT1, ISG15, MX1, and OAS1 indicate interferon response.",
    "QC note: doublet scores above 0.25 were removed.",
    "Batch note: donor and library chemistry can confound differential expression.",
] * 8)
too_much_context = large_context + "\n" + "\n".join(
    [f"Irrelevant lab inventory line {i}: freezer box metadata unrelated to expression." for i in range(120)]
)

contexts = {"no_context": "", "small_context": small_context, "large_context": large_context, "too_much_context": too_much_context}
question = "Why might IFIT1 and ISG15 be elevated, and what caveats should be checked?"
rows = []
for label, ctx in contexts.items():
    prompt = f"Context:\n{ctx}\n\nQuestion: {question}" if ctx else question
    start = time.perf_counter()
    response = OPENROUTER_CLIENT.chat.completions.create(
        model=DEFAULT_MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    usage = response.usage
    completion_tokens_details = usage.completion_tokens_details
    rows.append({"context": label, "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
    time.sleep(REQUEST_PAUSE_SECONDS)

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 6. Output parsing

This determines in what format the LLM produces the output, eg. plain text, JSON, pydantic. This is relevant in the context of agents because downstream processes that use LLM's output as input may require it in specific formats. Additionally, structured formats enforce strict validation checking.  

**Reflection Prompts**
- Compare free-form and structured output: which is easier to read, validate, and reuse?
- Observe what is lost when a rich answer has to fit into predefined fields.
- Decide when it is worth enforcing a rigid schema in a scientific pipeline.


In [4]:
QUESTION = "Interpret high FKBP5 and CRISPLD2 expression in dexamethasone-treated airway smooth muscle cells."


### i. Free text

Free text output is easy for humans to read but unreliable for software to process downstream.

**Reflection Prompts**
- Identify which parts of the answer are clear for humans but difficult to compare automatically.
- Note whether the model signals uncertainty or caveats without being forced to do so by a schema.
- Think about which criteria you would use to assign consistent scores to free-form answers.


In [ ]:
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": QUESTION}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
answer = response.choices[0].message.content
display(Markdown(answer))

print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:", usage.total_tokens)
print("reasoning_tokens:", completion_tokens_details.reasoning_tokens)
print("cost:", usage.cost)


### ii. JSON output

JSON output asks the model to return machine-readable fields. However, malformed JSON can still occur without validation or native structured output.


**Reflection Prompts**
- Check whether the JSON is valid and whether all fields are filled in informatively.
- Compare human readability with usefulness for an automated pipeline.
- Look for examples where the model follows the format but oversimplifies the content.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser, StrOutputParser

parser = JsonOutputParser()
prompt = f"Return valid JSON only with keys answer, confidence, caveats.\n\nQuestion: {QUESTION}"
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
answer = response.choices[0].message.content
parsed_response = parser.parse(answer)
print(parsed_response)

print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:", usage.total_tokens)
print("reasoning_tokens:", completion_tokens_details.reasoning_tokens)
print("cost:", usage.cost)


### iii. Pydantic parser

Pydantic parser validates types and required fields, making outputs usable directly in downstream Python code.

In this example, we can access the `parser.answer`, `parser.confidence`, and `parser.caveats` variables within Python, making it much easier to build reliable pipelines and AI agents. 


**Reflection Prompts**
- Observe which schema constraints help catch errors or ambiguity.
- Evaluate whether validation improves scientific quality or only formal compliance.
- Discuss which fields you would add to make the answer more useful in a lab setting.


In [ ]:
from pydantic import BaseModel, Field, ValidationError

class BioAnswer(BaseModel):
    answer: str
    confidence: Literal["low", "medium", "high"]
    caveats: list[str]

parser = PydanticOutputParser(pydantic_object=BioAnswer)

prompt = f"""
You are assisting with RNA-seq interpretation.
{parser.get_format_instructions()}
Question:
{QUESTION}
"""

start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
parsed_response = parser.parse(response.choices[0].message.content)
print(parsed_response)

print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:", usage.total_tokens)
print("reasoning_tokens:", completion_tokens_details.reasoning_tokens)
print("cost:", usage.cost)


Downstream use of pydantic parsing: routing the pipeline/agent based on confidence:

In [ ]:
if parsed_response.confidence == "high":
    print("✅ Proceed with pathway enrichment.")
elif parsed_response.confidence == "medium":
    print("⚠️ Check supporting literature before interpreting.")
else:
    print("❌ Collect more evidence before drawing conclusions.")

### iv. Markdown formatting

Markdown formatting controls the shape of the answer, such as plain text, tables, or bullet lists, which affects readability and downstream copy-paste usability.


**Reflection Prompts**
- Compare which format makes it easiest to quickly find results, caveats, and conclusions.
- Evaluate whether the table forces a more orderly comparison than the paragraph or bullet points.
- Ask which format you would use for personal notes, scientific reports, or machine-readable output.


In [ ]:
formats = {
    "plain_text": "Answer in one short paragraph.",
    "table": "Answer as a markdown table with columns Finding, Interpretation, Caveat.",
    "bullet_list": "Answer as concise bullet points.",
}
question = "Summarize how to interpret marker genes, QC metrics, and batch effects in scRNA-seq."

for label, instruction in formats.items():
    prompt = f"{instruction}\n\n{question}"
    start = time.perf_counter()
    response = OPENROUTER_CLIENT.chat.completions.create(
        model=DEFAULT_MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    usage = response.usage
    completion_tokens_details = usage.completion_tokens_details
    answer = response.choices[0].message.content
    display(Markdown(f"### {label}\n\n{answer}"))
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", usage.prompt_tokens)
    print("completion_tokens:", usage.completion_tokens)
    print("total_tokens:", usage.total_tokens)
    print("reasoning_tokens:", completion_tokens_details.reasoning_tokens)
    print("cost:", usage.cost)
    time.sleep(REQUEST_PAUSE_SECONDS)


## 7. Streaming

Streaming controls whether tokens are delivered incrementally, which affects user experience and perceived latency without necessarily changing the final answer.


**Reflection Prompts**
- Distinguish perceived latency from total time: which matters more for the final user?
- Observe whether seeing the answer as it is generated changes how you evaluate it.
- Think of scientific cases where streaming is useful and cases where it could be distracting.


In [ ]:
prompt = "Give a concise explanation of pseudobulk differential expression."

chunks = []
stream_usage = None
start = time.perf_counter()
for chunk in OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    stream=True,
    stream_options={"include_usage": True},
):
    if chunk.choices and chunk.choices[0].delta.content:
        text = chunk.choices[0].delta.content
        chunks.append(text)
        print(text, end="")
    if chunk.usage:
        stream_usage = chunk.usage

answer = "".join(chunks)
usage = stream_usage
completion_tokens_details = usage.completion_tokens_details if usage else None
print("\n\nstream_seconds:", round(time.perf_counter() - start, 3))

stream_df = pd.DataFrame([{
    "stream": True,
    "seconds": round(time.perf_counter() - start, 3),
    "prompt_tokens": getattr(usage, "prompt_tokens", None),
    "completion_tokens": getattr(usage, "completion_tokens", None),
    "total_tokens": getattr(usage, "total_tokens", None),
    "reasoning_tokens": getattr(completion_tokens_details, "reasoning_tokens", None),
    "cost": getattr(usage, "cost", None),
    "usage": usage,
    "answer": answer,
}])
stream_df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## Open ended exercise

Search OpenRouter for a model you want to try by exploring the models available and using filters, choose one model ID, then ask your own biological question while changing generation parameters such as `temperature`, `top_p`, and `max_tokens`.

Try to compare at least two settings and reflect on how the answer changes.

In [ ]:
MODEL = ""
QUESTION = ""
TEMPERATURE = 0
TOP_P = 0

response = OPENROUTER_CLIENT.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
    temperature=TEMPERATURE,
    top_p=TOP_P,
)

print(response.choices[0].message.content)